# Inspect Reports (Template)

This notebook helps you **inspect artifacts under `reports/`** produced by the training pipeline:
- `metrics.csv` (human-readable summary)
- `*_metrics.json` (per-model details)
- `*_oof.npy` (out-of-fold predictions)
- `*_test_pred.npy` (test predictions)

It also includes:
1) **OOF vs SalePrice** visualizations (scatter + residual distribution)  
2) A **model comparison overview** (interactive table + comparison plots)

> Run this notebook from the **repo root** (recommended). If you run it elsewhere, adjust `REPO_ROOT`.


In [ ]:
# =========================
# 0) Setup
# =========================
from pathlib import Path
import json
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

REPO_ROOT = Path('.')
DATA_DIR = REPO_ROOT / 'data'
REPORTS_DIR = REPO_ROOT / 'reports'

TRAIN_PATH = DATA_DIR / 'train.csv'

print('REPO_ROOT   :', REPO_ROOT.resolve())
print('DATA_DIR    :', DATA_DIR.resolve())
print('REPORTS_DIR :', REPORTS_DIR.resolve())

assert DATA_DIR.exists(), f"Missing {DATA_DIR}. Expected data/ with train.csv & test.csv"
assert REPORTS_DIR.exists(), f"Missing {REPORTS_DIR}. Run training to create reports/"
assert TRAIN_PATH.exists(), f"Missing {TRAIN_PATH}. Put Kaggle train.csv under data/"

train = pd.read_csv(TRAIN_PATH)
print('train shape:', train.shape)
train.head()


In [ ]:
# =========================
# 1) Quick directory inventory
# =========================
def ls_reports(reports_dir: Path):
    files = sorted([p for p in reports_dir.iterdir() if p.is_file()], key=lambda p: p.name.lower())
    df = pd.DataFrame({
        'file': [p.name for p in files],
        'size_kb': [round(p.stat().st_size/1024, 2) for p in files],
        'suffix': [p.suffix.lower() for p in files],
    })
    return df

inv = ls_reports(REPORTS_DIR)
inv


In [ ]:
# =========================
# 2) Read metrics.csv (if present)
# =========================
metrics_csv = REPORTS_DIR / 'metrics.csv'
if metrics_csv.exists():
    metrics = pd.read_csv(metrics_csv)
    print('metrics.csv:', metrics.shape)
    metrics.head(20)
else:
    print('metrics.csv not found. (This is OK if your train script only writes json/npy)')


In [ ]:
# =========================
# 3) Discover per-model artifacts
# =========================
# We infer model names from filenames like:
#   ridge_metrics.json, ridge_oof.npy, ridge_test_pred.npy
#   extratrees_metrics.json, ...
def infer_model_name(path: Path) -> str:
    name = path.stem
    for suf in ['_metrics', '_oof', '_test_pred', '_test_preds', '_oof_pred', '_oof_preds']:
        if name.endswith(suf):
            return name[: -len(suf)]
    return name

def gather_model_artifacts(reports_dir: Path):
    files = list(reports_dir.glob('*'))
    info = {}
    for p in files:
        if not p.is_file():
            continue
        if p.name in ('metrics.csv', 'cv_summary.json'):
            continue
        if p.suffix.lower() not in ('.json', '.npy', '.csv', '.parquet'):
            continue
        m = infer_model_name(p)
        info.setdefault(m, {'model': m})
        if p.name.endswith('_metrics.json'):
            info[m]['metrics_json'] = p.as_posix()
        if p.name.endswith('_oof.npy'):
            info[m]['oof_npy'] = p.as_posix()
        if p.name.endswith('_test_pred.npy'):
            info[m]['test_pred_npy'] = p.as_posix()
    return pd.DataFrame(list(info.values())).sort_values('model')

art = gather_model_artifacts(REPORTS_DIR)
art


In [ ]:
# =========================
# 4) Load *_metrics.json into a comparison table
# =========================
def load_json(path_str: str):
    p = Path(path_str)
    with open(p, 'r', encoding='utf-8') as f:
        return json.load(f)

rows = []
if not art.empty and 'metrics_json' in art.columns:
    for _, r in art.dropna(subset=['metrics_json']).iterrows():
        m = r['model']
        d = load_json(r['metrics_json'])
        row = {'model': m, '_raw': d}
        # try common keys
        for k in ['cv_rmse', 'rmse', 'rmse_mean', 'mean_rmse', 'std_rmse']:
            if k in d:
                row[k] = d[k]
        # folds often under 'folds' or 'fold_scores'
        for k in ['folds', 'fold_scores']:
            if k in d:
                row[k] = d[k]
        rows.append(row)

metrics_json_df = pd.DataFrame(rows)

def extract_cv_rmse(row):
    for k in ['cv_rmse', 'rmse', 'rmse_mean', 'mean_rmse']:
        if k in row and pd.notna(row[k]):
            try:
                return float(row[k])
            except:
                pass
    return np.nan

if not metrics_json_df.empty:
    metrics_json_df['cv_rmse_guess'] = metrics_json_df.apply(lambda x: extract_cv_rmse(x.to_dict()), axis=1)
    metrics_json_df[['model', 'cv_rmse_guess']].sort_values('cv_rmse_guess')
else:
    print('No *_metrics.json found yet. Train more models to generate them.')


In [ ]:
# =========================
# 5) Model comparison plot (from *_metrics.json)
# =========================
if 'metrics_json_df' in globals() and not metrics_json_df.empty and metrics_json_df['cv_rmse_guess'].notna().any():
    df_plot = metrics_json_df[['model', 'cv_rmse_guess']].dropna().sort_values('cv_rmse_guess')
    fig = px.bar(df_plot, x='model', y='cv_rmse_guess', title='CV RMSE (log-space) by Model (from *_metrics.json)')
    fig.update_layout(xaxis_tickangle=-30)
    fig.show()
else:
    print('Not enough metrics to plot yet.')


In [ ]:
# =========================
# 6) Alternate comparison plot (from metrics.csv)
# =========================
if 'metrics' in globals() and isinstance(metrics, pd.DataFrame) and not metrics.empty:
    score_col = None
    for c in metrics.columns:
        if 'rmse' in c.lower():
            score_col = c
            break
    if score_col is None and metrics.shape[1] >= 2:
        score_col = metrics.columns[1]

    if score_col is not None:
        df_plot = metrics.sort_values(score_col)
        fig = px.bar(df_plot, x=df_plot.columns[0], y=score_col, title=f"Model comparison from metrics.csv ({score_col})")
        fig.update_layout(xaxis_tickangle=-30)
        fig.show()
    metrics.head()
else:
    print('metrics.csv not loaded (or empty).')


## OOF + SalePrice Visualization

OOF predictions are typically in **log1p space** (depending on your pipeline).  
We will:
- infer prediction scale (raw vs log) by magnitude
- plot **OOF Pred vs True** and **residuals**
- show worst cases


In [ ]:
# =========================
# 7) Select a model for OOF inspection
# =========================
available = art['model'].tolist() if not art.empty else []
available


In [ ]:
MODEL = available[0] if available else None
print('Selected MODEL =', MODEL)

if MODEL is None:
    raise ValueError('No models detected in reports/. Train at least one model.')

row = art[art['model'] == MODEL].iloc[0]
row


In [ ]:
# =========================
# 8) Load OOF and build analysis DataFrame
# =========================
def infer_pred_scale(y_pred):
    # Typical SalePrice raw median > 1000; log1p median ~ 12
    return 'log' if np.nanmedian(y_pred) < 50 else 'raw'

def make_oof_df(train_df: pd.DataFrame, oof_pred: np.ndarray):
    ids = train_df['Id'].to_numpy() if 'Id' in train_df.columns else np.arange(len(train_df))

    y_true_raw = train_df['SalePrice'].to_numpy(dtype=float)
    y_true_log = np.log1p(y_true_raw)

    scale = infer_pred_scale(oof_pred)
    if scale == 'log':
        y_pred_log = oof_pred.astype(float).ravel()
        y_pred_raw = np.expm1(y_pred_log)
    else:
        y_pred_raw = oof_pred.astype(float).ravel()
        y_pred_log = np.log1p(np.maximum(y_pred_raw, 0))

    df = pd.DataFrame({
        'Id': ids,
        'y_true_raw': y_true_raw,
        'y_pred_raw': y_pred_raw,
        'y_true_log': y_true_log,
        'y_pred_log': y_pred_log,
    })
    df['residual_raw'] = df['y_true_raw'] - df['y_pred_raw']
    df['residual_log'] = df['y_true_log'] - df['y_pred_log']
    df['abs_error_raw'] = np.abs(df['residual_raw'])
    df['abs_error_log'] = np.abs(df['residual_log'])
    df['ape_raw'] = df['abs_error_raw'] / np.maximum(df['y_true_raw'], 1.0)
    return df

oof_path = Path(row['oof_npy']) if 'oof_npy' in row and pd.notna(row['oof_npy']) else None
assert oof_path is not None and oof_path.exists(), f"Missing OOF file for {MODEL}"
oof = np.load(oof_path)

print('Loaded:', oof_path.name, 'shape=', oof.shape)
df_oof = make_oof_df(train, oof)
df_oof.head()


In [ ]:
# =========================
# 9) OOF Pred vs True (log space)
# =========================
fig = px.scatter(
    df_oof,
    x='y_pred_log',
    y='y_true_log',
    hover_data=['Id', 'y_true_raw', 'y_pred_raw', 'ape_raw'],
    title=f"OOF Pred vs True (log1p) — {MODEL}",
)
mn = float(min(df_oof['y_pred_log'].min(), df_oof['y_true_log'].min()))
mx = float(max(df_oof['y_pred_log'].max(), df_oof['y_true_log'].max()))
fig.add_trace(go.Scatter(x=[mn, mx], y=[mn, mx], mode='lines', name='y=x'))
fig.show()


In [ ]:
# =========================
# 10) Residual vs Prediction (raw)
# =========================
fig = px.scatter(
    df_oof,
    x='y_pred_raw',
    y='residual_raw',
    hover_data=['Id', 'y_true_raw', 'y_pred_raw', 'ape_raw'],
    title=f"Residual vs Prediction (raw) — {MODEL}",
)
fig.add_hline(y=0)
fig.show()


In [ ]:
# =========================
# 11) Residual distributions (raw + log)
# =========================
px.histogram(df_oof, x='residual_log', nbins=60, title=f"Residual Distribution (log1p) — {MODEL}").show()
px.histogram(df_oof, x='residual_raw', nbins=60, title=f"Residual Distribution (raw) — {MODEL}").show()


In [ ]:
# =========================
# 12) Top error cases
# =========================
df_oof.sort_values('abs_error_raw', ascending=False).head(25)[
    ['Id', 'y_true_raw', 'y_pred_raw', 'abs_error_raw', 'ape_raw', 'y_true_log', 'y_pred_log']
].reset_index(drop=True)


## Bonus: Segment Error by a Categorical Feature (e.g., Neighborhood)

This helps identify systematic bias across groups.


In [ ]:
# =========================
# 13) Segment MAE by a categorical feature
# =========================
features = train.drop(columns=['SalePrice'], errors='ignore').copy()
if 'Id' not in features.columns:
    features['Id'] = np.arange(len(features))

df_full = df_oof.merge(features, on='Id', how='left')

cat_col = 'Neighborhood' if 'Neighborhood' in df_full.columns else None
cat_col


In [ ]:
if cat_col is None:
    print('Neighborhood not found. Choose another categorical column from:')
    print([c for c in df_full.columns if df_full[c].dtype == 'object'][:30])
else:
    g = df_full.groupby(cat_col, observed=True).agg(
        n=('Id', 'count'),
        mae=('abs_error_raw', 'mean'),
        mape=('ape_raw', 'mean'),
        median_price=('y_true_raw', 'median'),
    ).reset_index().sort_values('mae', ascending=False)

    g_top = g.head(25)
    px.bar(
        g_top,
        x=cat_col,
        y='mae',
        hover_data=['n', 'mape', 'median_price'],
        title=f"Top 25 {cat_col} by MAE (OOF) — {MODEL}",
    ).update_layout(xaxis_tickangle=-45).show()

    g_top.reset_index(drop=True)
